# Convert Student Paper Index spreadsheet to Invenio or DataCite json.

In [ ]:
import pandas as pd
import json
import os

In [ ]:
dir = '/Users/metadatagamechanger/Repositories/MooreaStudentPapers/'
input = dir + 'data/index_to_moorea_class_projects_1992-2022.csv'
input_df = pd.read_csv(input)
input_df.head()

In [ ]:
def rowToMetadata(i:                int,  # row index in dataframe
                  input_df,
                  paperIndex:       int,  # index of paper in publication year
                  metadataType:     str,  # type of metaata (invenio or dataCite)
                  makeTemplate:     bool  # if true, add template items to the metadata
                  ):
        
    prefix = '10.60950'

    if metadataType == 'dataCite':

        metadata_d = {
            #
            # In order to publish a new DOI, the "id" element should not be included in the metadata. 
            # The DOI is specified in the "doi" element of the attributes section. 
            # The landing page must also be specified in the "url" element of the attributes section.
            # The event attribute = "publish" makes the DOI findable immediately upon registration. 
            # If the event attribute is not included, the DOI will be registered but not active until
            # the publisher makes it active. 
            # 
            "data": {
            #    "id": "",
                "type": "dois",
                "attributes": {
#                    "event": "publish",            # this was included for metadata creation, not needed for updates
                    "doi": "",
                    "url": ""
                }
            }
        }
        #
        #   Mandatory Fields: creators, titles, publisher, publicationYear, resourceTypeGeneral
        #
        #   The metadata schema includes a "creators" element that is a list of creator objects. 
        #   Each creator object includes the creator's name, name type (Personal or Organizational), 
        #   affiliation, and optionally an identifier such as ORCID.
        #
        metadata_d['data']['attributes']['creators'] = []              # Add creators to metadata dictionary, initialize as empty list

        for p, person in enumerate(input_df.at[i, 'Authors, Primary'].split(';')):   # Loop the authors and make creators in the metadata dictionary
            person = person.replace('Dr. ','')
            if ',' in str(person):
                toks = person.split(',')
            else:
                toks = person.split(' ,')

            if len(toks) < 2 or len(toks) > 4:
                continue

            family = toks[0].replace(',','') if ',' in str(person) else toks[1]
            family = family.strip()

            given = " ".join(toks[1:]) if ',' in str(person) else toks[0]
            given = given.strip()

            metadata_d['data']['attributes']['creators'].append({
                                                        'name': f"{given} {family}",    # name is mandatory for all creators
                                                        'nameType': 'Personal',
                                                        'givenName':   given,
                                                        'familyName':  family,
                                                        'affiliation': [{
                                                                        "name": "University of California, Berkeley",
                                                                        "affiliationIdentifier": "https://ror.org/01an7q238",
                                                                        "affiliationIdentifierScheme": "ROR",
                                                                        "schemeUri": "https://ror.org/"
                                                                        }]
                                                        })
            if makeTemplate:
                #
                # The metadata schema allows for identifiers such as ORCID to be included for creators. 
                # If the identifier is not known, it can be left out or it can be included with a standard 
                # placeholder value such as ":unkn".
                #
                if 'nameIdentifiers' not in metadata_d['data']['attributes']['creators'][-1]:    # add nameIdentifiers if it doesn't exist, initialize as empty list
                    metadata_d['data']['attributes']['creators'][-1]['nameIdentifiers'] = []

                metadata_d['data']['attributes']['creators'][-1]['nameIdentifiers'].append({      # add nameIdentifier dictionary to list
                                                                        "schemeUri": "https://orcid.org",
                                                                        "nameIdentifier": ":unkn",
                                                                        "nameIdentifierScheme": "ORCID"
                                                                        })
            #
            # The metadata schema includes an optional "contributors" element that is a list of contributor objects. 
            # Each contributor object includes the contributor's name, name type (Personal or Organizational), 
            # affiliation, and optionally an identifier such as ORCID. The contributorType is a required
            # element of the contributor object that identifies the type of contribution. 
            # In this case, all authors are assigned the contributorType "RightsHolder" because the content 
            # is copyrighted (the metadata includes a copyrighted date).
            #
            if 'contributors' not in metadata_d['data']['attributes']:                  # add contributors if it doesn't exist
                metadata_d['data']['attributes']['contributors'] = []                   # it is an empty list.
            
            metadata_d['data']['attributes']['contributors'].append({                   # add each author as a contributor with type "RightsHolder"
                                                        'name': f"{given} {family}",    # name is mandatory for all contributors
                                                        'nameType': 'Personal',
                                                        'givenName':   given,
                                                        'familyName':  family,
                                                        'affiliation': [{
                                                                        "name": "University of California, Berkeley",
                                                                        "affiliationIdentifier": "https://ror.org/01an7q238",
                                                                        "affiliationIdentifierScheme": "ROR",
                                                                        "schemeUri": "https://ror.org/"
                                                                        }],
                                                        "contributorType": "RightsHolder"
                                                        })
            
            if makeTemplate:
            #
            # People and organizations can have identifiers such as ORCID for people and and ROR for organizations. 
            # If the identifier is known, it should be added here. If not known, it can be left out or it can be included 
            # with a standard placeholder value such as ":unkn".
            #
                if 'nameIdentifiers' not in metadata_d['data']['attributes']['contributors'][-1]:    # add nameIdentifiers if it doesn't exist, initialize as empty list
                    metadata_d['data']['attributes']['contributors'][-1]['nameIdentifiers'] = []

                metadata_d['data']['attributes']['contributors'][-1]['nameIdentifiers'].append(      # add nameIdentifier dictionary to list
                                                                                        {
                                                                                        "schemeUri": "https://orcid.org",
                                                                                        "nameIdentifier": ":unkn",
                                                                                        "nameIdentifierScheme": "ORCID"
                                                                                        })
#
        # Create a unique identifier for each record using a combination of the iPlaces DOI prefix, the abbreviation for the class
        # (Biology and Geology of Tropical Islands), the publication year, the paper index, and creator names and initials.
        # The creator names are concatenated together with periods, and spaces and special characters are removed or 
        # replaced to ensure the identifier is valid.
        #
        identifier = f"{'.'.join([ x['familyName'].replace(' ','')+'.'+ ''.join(c for c in x['givenName'] if not c.islower()).replace(' ','').replace('.','') for x in metadata_d['data']['attributes']['creators'] ])}"
        identifier = identifier.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        identifier = f"{prefix}/bgtl.{str(input_df.at[i, 'Pub Year'])}.{paperIndex}.{identifier}"
        #
        # to create a new doi do not include the id element
        # metadata_d['id'] = identifier
        #
        metadata_d['data']['attributes']['doi'] = identifier
        #
        # if a new DOI is being created it needs a landing page url. 
        # In this case the landing page is set to the commons page for the DOI.
        # The commons URL is a placeholder that ensures the DOI will resolve when registered and published.
        # A different landing page can be added later, but it must be a valid URL. 
        #
        metadata_d['data']['attributes']['url'] = f'https://commons.datacite.org/doi.org/{identifier}'                  # set landing page to commons page
        #
        # The title is a mandatory field in the metadata schema.
        # The metadata schema allows for multiple titles with different languages.
        # In this case there is only one title per record.
        # It is specified using a dictionary with the title and language. 
        # In this case, the language is set to English ("en") for all records.
        # The title is only added if it is not empty or "nan" in the input spreadsheet.
        #
        if len(str(input_df.at[i, 'Title Primary'])) > 0 and str(input_df.at[i, 'Title Primary']) != 'nan':
            metadata_d['data']['attributes']['titles'] = [{"lang": "en", "title": input_df.at[i, 'Title Primary']}]
        #
        # The publication year is a mandatory field in the metadata schema. 
        # It is specified in the "publicationYear" element and must be a four-digit year.
        #
        metadata_d['data']['attributes']['publicationYear']    = input_df.at[i, 'Pub Year']
        # The publisher is a mandatory field in the metadata schema. 
        # It is specified in the "publisher" element which is a dictionary that can include
        # the publisher name, identifier, and identifier scheme.
        #
        metadata_d['data']['attributes']['publisher']          = {
                                                        "name": "University of California, Berkeley",
                                                        "publisherIdentifier": "https://ror.org/01an7q238",
                                                        "publisherIdentifierScheme": "ROR",
                                                        "schemeUri": "https://ror.org/",
                                                        "lang": "en"
                                                        }
        #
        # The resource type general is a mandatory field in the metadata schema.
        # The resource type is specified in the "types" element which is a dictionary 
        # that can include the general type resourceTypeGeneral selected from a shared list,
        # and a more specific type as resourceType (free text). 
        # 
        metadata_d['data']['attributes']['types'] = {
                                        "resourceTypeGeneral": "Text",
                                        "resourceType": "Student paper from Biology and Geology of Tropical Islands class at University of California, Berkeley"
                                    }
        #
        # This is the end of the mandatory metadata fields. 
        # The remaining metadata elements are recommended or optional. 
        # They are included if the information is available in the input spreadsheet.
        #
        # The metadata schema includes an optional "descriptions" element that is a list of description objects.
        # Each description object includes the description, description type, and language.
        # In this case, the description type is set to "Abstract" and the language is set to English ("en") for all records.
        # The Abstract is only added if it is not empty.
        #
        if len(str(input_df.at[i, 'Abstract'])) > 0 and str(input_df.at[i, 'Abstract']) != 'nan' and str(input_df.at[i, 'Abstract']) != ' ' :
            metadata_d['data']['attributes']['descriptions'] = [{
                                                        "lang": "en", 
                                                        "description": input_df.at[i, 'Abstract'],
                                                        "descriptionType": "Abstract"
                                                        }]
        else:           # if no abstract is available when record is being created, add a description 
                        # with a template value that can be filled in later
            if makeTemplate:
                metadata_d['data']['attributes']['descriptions'] = [{
                                                        "lang": "en", 
                                                        "description": ":unkn",
                                                        "descriptionType": "Abstract"
                                                        }]
        #
        # The metadata schema includes an optional "dates" element that is a list of date objects.
        # Each date object includes the date, date type, and optionally a date information element that
        # can include information about the date such as the format. 
        # In this case, the date type is set to "Copyrighted"
        metadata_d['data']['attributes']['dates'] = []
        metadata_d['data']['attributes']['dates'].append({
                                            "date": str(input_df.at[i, 'Pub Year']),
                                            "dateType": "Copyrighted"
                                        })
        #
        # The Coverage date type is the time that data were collected for the paper, which is the publication year 
        # between August and December in this case. The string year-08/year-12 is used to indicate this in the date element.
        # see https://www.ukoln.ac.uk/metadata/dcmi/collection-RKMS-ISO8601/ for details on date standard
        #
        metadata_d['data']['attributes']['dates'].append({
                                            "date": f"{str(input_df.at[i, 'Pub Year'])}-08/{str(input_df.at[i, 'Pub Year'])}-12",
                                            "dateType": "Coverage"
                                        })
        #
        # The metadata schema includes an optional "contributors" element that is a list of contributor objects.
        # Each contributor object includes the contributor's name, name type (Personal or Organizational),
        # affiliation, and optionally an identifier such as ORCID. The contributorType is a required
        # element of the contributor object that identifies the type of contribution. 
        # In this case, Gump South Pacific Research Station is included as a contributor with the 
        # contributorType "Sponsor" because Gump is the sponsor of the class project. This element
        # is added to the metadata to connect the object to the field station.
        #
        contributor_Gump = {
                            "nameType": "Organizational",
                            "name": "Gump South Pacific Research Station",
                            "affiliation": [{
                                "name": "Gump South Pacific Research Station",
                                "affiliationIdentifier": "https://ror.org/04sk0et52",
                                "affiliationIdentifierScheme": "ROR",
                                "schemeUri": "https://ror.org/",
                                "lang": "en"
                            }],
                            "contributorType": "Sponsor",
                            "nameIdentifiers": [{
                                    "nameIdentifier": "https://ror.org/04sk0et52",
                                    "nameIdentifierScheme": "ROR",
                                    "schemeUri": "https://ror.org/"
                                }]
                            }
        
        if 'contributors' not in metadata_d['data']['attributes']:          # just in case contributors doesn't exist, add it as an empty list
            metadata_d['data']['attributes']['contributors'] = []

        metadata_d['data']['attributes']['contributors'].append(contributor_Gump)     # add Gump as a contributor with type "Sponsor"
        #
        # The index spreadsheet includes a column named "Keywords" that includes fairly high-level keywords about the content of the paper.
        # The metadata includes an element named "subjects" that is a list of subject objects.
        # Each subject object includes the subject and optionally a subjectScheme that identifies keywords of a specific type.
        # The keywords in the spreadsheet are added as subjects in the metadata without a subjectScheme, which differentiates
        # them from the organism and where keywords that are added below with a subjectScheme.
        #
        if len(str(input_df.at[i, 'Keywords'])) > 0 and str(input_df.at[i, 'Keywords']) != 'nan':
            keywords_l = str(input_df.at[i, 'Keywords']).split(';')              # some keywords are delimited with ";"
            if len(keywords_l) == 1:
                keywords_l = str(input_df.at[i, 'Keywords']).split(',')          # some keywords are delimited with ","

            for subject in keywords_l:
                subject = subject.strip()
                if len(subject) > 0:
                    if 'subjects' not in metadata_d['data']['attributes']:
                        metadata_d['data']['attributes']['subjects'] = []
                    metadata_d['data']['attributes']['subjects'].append({'subject': subject})
        #
        # The index spreadsheet includes a column named "organism" that includes fairly high-level organism keywords
        # The metadata includes an element named subjectScheme that identifies keywords of a specific type.
        # In this case that is set to "organism" to differentiate these keywords from the general set
        #  
        if all(['organism' in input_df.columns,                         # organiam column exists in input spreadsheet,
                len(str(input_df.at[i, 'organism'])) > 0,               # and it is not empty or "nan" for this record
                str(input_df.at[i, 'organism']) != 'nan']):
            
            if 'subjects' not in metadata_d['data']['attributes']:
                metadata_d['data']['attributes']['subjects'] = []

            for subject in str(input_df.at[i, 'organism']).split(','):
                metadata_d['data']['attributes']['subjects'].append({'subject': subject.strip(), 'subjectScheme':'organism'})

        if makeTemplate:                                                # add subject dictionary with scheme = 'organism' 
                                                                        # and subject = ":unkn" so new organism information
                                                                        # can be added later if it is not available in the input spreadsheet
            if not 'subjects' in metadata_d['data']['attributes']:
                metadata_d['data']['attributes']['subjects'] = []

            metadata_d['data']['attributes']['subjects'].append({'subject': ':unkn', 'subjectScheme':'organism'})
        #
        # The index spreadsheet includes a column named "where" that includes fairly high-level location keywords
        # The metadata includes an element named geoLocation that identifies keywords that are locations.
        # In this case that is added to the geoLocations list to differentiate these keywords from the general set.
        #  
        if all(['where' in input_df.columns, 
                len(str(input_df.at[i, 'where'])),
                str(input_df.at[i, 'where']) != 'nan']):
            if 'geoLocations' not in metadata_d['data']['attributes']:
                metadata_d['data']['attributes']['geoLocations'] = []
            metadata_d['data']['attributes']['geoLocations'].append({'geoLocationPlace': input_df.at[i, 'where']})

        if makeTemplate:                                                # add geolocation dictionary with scheme = 'where' 
                                                                        # and subject = ":unkn" so new location information
                                                                        # can be added later if it is not available in the input spreadsheet
            if not 'geoLocations' in metadata_d['data']['attributes']:
                metadata_d['data']['attributes']['geoLocations'] = []

            metadata_d['data']['attributes']['geoLocations'].append({'geoLocationPlace': ':unkn'})

        #
        # The metadata schema includes an optional "relatedIdentifiers" element that is a list of related identifier objects.
        # Each related identifier object includes the related identifier, relation type, related identifier type, and
        # resource type general of the related object.
        # In this case, a related identifier is included with the relation type "IsPartOf" that connects the paper to
        # a project identifier for the class project for that publication year. 
        # This element is added to the metadata to connect the object to the project.
        # The default related identifier is a DOI for the entire class project, i.e. the parent project of all classes.
        # It needs to be replaced with the identifier for the project for the specific publication year, which is not active yet. 
        #
        relatedIsPartOf = f"10.60950/956fad7b-311c-4250-bebc-4e43f0c2ffdc"
        metadata_d['data']['attributes']['relatedIdentifiers'] = [{'relatedIdentifier': relatedIsPartOf, 
                                                        'relationType':          'IsPartOf',
                                                        "resourceTypeGeneral":   'Project',
                                                        "relatedIdentifierType": "DOI"}]
        #
        # related identifiers can also be used to connect the paper to related works such as a published 
        # version of the paper or a dataset associated with the paper. These related identifiers can also
        # connect to other student papers from the same class. They can also connect to related works
        # such as papers from the scientific literature. Those are not linked yet.
        #
        if makeTemplate:
            if 'relatedIdentifiers' not in metadata_d['data']['attributes']:    # add relatedIdentifiers if it doesn't exist, 
                                                                                # initialize as empty list
                    metadata_d['data']['attributes']['relatedIdentifiers'] = []

            metadata_d['data']['attributes']['relatedIdentifiers'].append(      # add template relatedIdentifiers dictionary to list
                                                                {'relatedIdentifier':       ':unkn', 
                                                                'relationType':             'Cites',
                                                                "resourceTypeGeneral":      'Text',
                                                                "relatedIdentifierType":    'DOI'}
                                                                )

        metadata_d['data']['attributes']['version'] = "1"

        metadata_d['data']['attributes']['rightsList'] = [{
                                                        "lang": "en",
                                                        "rights": "Creative Commons Attribution 4.0 International",
                                                        "rightsUri": "https://creativecommons.org/licenses/by/4.0/legalcode",
                                                        "schemeUri": "https://spdx.org/licenses/",
                                                        "rightsIdentifier": "cc-by-4.0",
                                                        "rightsIdentifierScheme": "SPDX"
                                                    }]

        #fileName = f"{'_'.join([ x['familyName'].replace(' ','')+'_'+ x['givenName'].replace(' ','').replace('.','') for x in metadata_d['data']['attributes']['creators'] ])}.json"
        fileName = identifier.replace('/','_')
        fileName = fileName.replace('/','-').replace(':','-').replace('?','-').replace('"','-').replace('\'','-').replace(',','').replace('(','').replace(')','').replace('\n','_')
        fileName = f'{str(metadata_d["data"]["attributes"]["publicationYear"])}/{fileName}'
  
    return metadata_d, fileName


## Create metadata records.
See https://datacite-metadata-schema.readthedocs.io/en/4.7/ 
or https://inveniordm.docs.cern.ch/reference/metadata/ for reference

In [ ]:
currentYear = 0                                             # initialize publication year variable, will be updated for each record in loop below
paperIndex  = 0                                             # initialize paper index variable, will be updated for each record in loop below
fileName    = ""                                            # initialize fileName variable
makeTemplate = True                                         # set makeTemplate to True to write out template items
                                                            # to be filled in later, set to False to only include 
                                                            # metadata elements for which there is information in 
                                                            # the input spreadsheet
for i in input_df.index:                                    # loop input rows (one paper/metadata record per row)

    pubYear = input_df.at[i, 'Pub Year']                # update publication year for current record
    if pubYear != currentYear:                          # new year, initialize paperIndex
        paperIndex = 1
        currentYear = pubYear
    else:                                               # same year as previous record, increment paperIndex
        paperIndex += 1

    metadata_d, fileName = rowToMetadata(i, input_df, paperIndex, metadataType='dataCite', makeTemplate=makeTemplate)     # create metadata dictionary for current record
    output = f'{dir}metadata/v1/{fileName}.json'
    
    os.makedirs(os.path.dirname(output), exist_ok=True)
    print(f"writing metadata to {'/'.join(output.split('/')[-2:])}")
    with open(output, 'w') as f:
        json.dump(metadata_d, f, indent=2, default=str)
